# Data Quality Checker — SQL Server + Great Expectations (validación en memoria, por categoría de columna)

Este notebook:

1. Extrae cada tabla de SQL Server con **una sola query** (`SELECT * FROM tabla`).
2. Valida el DataFrame resultante con Great Expectations usando un **PandasDatasource** —
   la validación corre en memoria, sin generar carga adicional sobre el servidor.
3. Clasifica automáticamente cada columna en una **categoría** (personal, código, fecha, monto,
   general) y aplica un conjunto de expectativas específico por categoría.
4. Genera el reporte HTML (Data Docs) con el detalle por campo.

**Requisitos previos:**
- Driver ODBC de SQL Server instalado (ODBC Driver 17 o 18 for SQL Server)
- Conexión configurada (SQL Auth o Windows Auth — ver celda de configuración)


## 1. Instalación de dependencias

Ejecutar solo la primera vez.

In [ ]:
# %pip install great_expectations pyodbc sqlalchemy pandas -q

## 2. Imports

In [ ]:
import os
import re
import pandas as pd
import great_expectations as gx
from great_expectations import ExpectationSuite, ValidationDefinition, Checkpoint
from sqlalchemy import create_engine, inspect

## 3. Configuración de conexión

Ajusta estos valores según tu entorno.

- **SQL Authentication**: define `MSSQL_USER` y `MSSQL_PASSWORD` como variables de entorno.
- **Windows Authentication**: deja `USE_WINDOWS_AUTH = True`, no necesitas usuario/contraseña.


In [ ]:
HOST = "tu_servidor"          # p.ej. "localhost" o "10.0.0.5"
DATABASE = "tu_base"
SCHEMA = "dbo"
PORT = 1433
DRIVER = "ODBC Driver 17 for SQL Server"   # cambia a 18 si es lo que tienes instalado

USE_WINDOWS_AUTH = False   # True = autenticación integrada de Windows

# Tablas a excluir del análisis (ej. tablas de log, staging, etc.)
EXCLUDE_TABLES = []

## 4. Motor de conexión (SQLAlchemy)

Se usa únicamente para la extracción inicial de cada tabla (una query por tabla).

In [ ]:
if USE_WINDOWS_AUTH:
    conn_str = (
        f"mssql+pyodbc://@{HOST}/{DATABASE}"
        f"?driver={DRIVER.replace(' ', '+')}"
        f"&trusted_connection=yes"
    )
else:
    conn_str = (
        f"mssql+pyodbc://{os.environ['MSSQL_USER']}:{os.environ['MSSQL_PASSWORD']}"
        f"@{HOST}:{PORT}/{DATABASE}?driver={DRIVER.replace(' ', '+')}"
    )

engine = create_engine(conn_str)
inspector = inspect(engine)

tablas = [t for t in inspector.get_table_names(schema=SCHEMA) if t not in EXCLUDE_TABLES]
print(f"{len(tablas)} tablas encontradas:")
tablas

## 5. Crear el Data Context de Great Expectations

In [ ]:
context = gx.get_context()  # contexto efímero en memoria
# Si prefieres que la configuración persista en disco entre sesiones, usa:
# context = gx.get_context(mode="file")

## 6. Datasource de Pandas

Este datasource valida DataFrames en memoria — no se conecta a SQL Server durante la
validación. La única query a la base ocurre al extraer cada tabla (celda siguiente, dentro
del loop principal).

In [ ]:
data_source = context.data_sources.add_pandas(name="pandas_datasource")

## 7. Clasificación de columnas por categoría

Cada columna se clasifica según coincidencias de palabras clave en su nombre. Ajusta las
listas según cómo estén nombradas tus columnas reales. Cualquier columna que no matchee
ninguna categoría cae en `"general"`.

In [ ]:
COLUMN_CATEGORIES = {
    "personal": ["nombre", "apellido", "razon_social", "cliente", "direccion", "telefono", "email"],
    "codigo": ["id", "codigo", "cod_", "_key", "nit", "ci"],
    "fecha": ["fecha", "date", "_at", "created", "updated"],
    "monto": ["monto", "precio", "total", "importe", "saldo"],
}


def clasificar_columna(nombre_columna: str) -> str:
    """Devuelve la categoría de la columna según sus palabras clave, o 'general' si no matchea nada."""
    nombre_lower = nombre_columna.lower()
    for categoria, keywords in COLUMN_CATEGORIES.items():
        if any(kw in nombre_lower for kw in keywords):
            return categoria
    return "general"

## 8. Expectativas por categoría

Cada categoría tiene su propio conjunto de reglas de calidad, adecuadas a su naturaleza
(ej. no se exige unicidad en nombres, sí en códigos; se valida rango de fechas, etc.).

In [ ]:
REGEX_NOMBRE_VALIDO = r"^[A-Za-zÁÉÍÓÚÑáéíóúñ]+(\s[A-Za-zÁÉÍÓÚÑáéíóúñ]+)*$"
REGEX_CODIGO_VALIDO = r"^[A-Za-z0-9\-_]+$"  # sin espacios ni símbolos raros


def expectations_personal(suite, col_name, dtype):
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column=col_name))
    if dtype == "object":
        suite.add_expectation(
            gx.expectations.ExpectColumnValuesToMatchRegex(
                column=col_name, regex=REGEX_NOMBRE_VALIDO, mostly=0.98
            )
        )


def expectations_codigo(suite, col_name, dtype):
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column=col_name))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column=col_name))
    if dtype == "object":
        suite.add_expectation(
            gx.expectations.ExpectColumnValuesToMatchRegex(
                column=col_name, regex=REGEX_CODIGO_VALIDO, mostly=1.0
            )
        )


def expectations_fecha(suite, col_name, dtype):
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column=col_name, mostly=0.95))
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToBeBetween(
            column=col_name, min_value="1900-01-01", max_value="2100-01-01"
        )
    )


def expectations_monto(suite, col_name, dtype):
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column=col_name))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column=col_name, min_value=0))


def expectations_general(suite, col_name, dtype):
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column=col_name, mostly=0.9))


# Registro: categoría -> función que agrega las expectativas correspondientes
EXPECTATION_BUILDERS = {
    "personal": expectations_personal,
    "codigo": expectations_codigo,
    "fecha": expectations_fecha,
    "monto": expectations_monto,
    "general": expectations_general,
}

## 9. Construcción dinámica de la suite, según categoría de cada columna

In [ ]:
def construir_suite(context, nombre_tabla, df):
    """Crea (o reemplaza) una Expectation Suite para una tabla, aplicando el conjunto de
    expectativas correspondiente a la categoría de cada columna."""
    suite_name = f"suite_{nombre_tabla}"
    suite = context.suites.add_or_update(ExpectationSuite(name=suite_name))

    clasificacion_columnas = {}  # para el resumen/auditoría posterior

    for col_name in df.columns:
        categoria = clasificar_columna(col_name)
        dtype = str(df[col_name].dtype)
        builder = EXPECTATION_BUILDERS[categoria]
        builder(suite, col_name, dtype)
        clasificacion_columnas[col_name] = categoria

    return suite, clasificacion_columnas

## 10. Recorrer todas las tablas: extraer, clasificar, validar

Por cada tabla: una sola query a SQL Server, clasificación de columnas, suite dinámica,
validación en memoria vía checkpoint, y registro del resumen.

In [ ]:
resumen_resultados = []

for nombre_tabla in tablas:
    print(f"Procesando tabla: {nombre_tabla}")

    # --- Única consulta a SQL Server para esta tabla ---
    query = f"SELECT * FROM {SCHEMA}.[{nombre_tabla}]"
    df = pd.read_sql(query, engine)

    # --- Data Asset en memoria (sin conexión a SQL) ---
    asset_name = f"asset_{nombre_tabla}"
    try:
        data_asset = data_source.get_asset(asset_name)
    except LookupError:
        data_asset = data_source.add_dataframe_asset(name=asset_name)

    batch_def_name = f"batch_{nombre_tabla}"
    try:
        batch_definition = data_asset.get_batch_definition(batch_def_name)
    except (LookupError, KeyError):
        batch_definition = data_asset.add_batch_definition_whole_dataframe(batch_def_name)

    batch_parameters = {"dataframe": df}

    # --- Suite dinámica por categoría de columna ---
    suite, clasificacion_columnas = construir_suite(context, nombre_tabla, df)

    # --- Validation Definition + Checkpoint ---
    validation_definition = context.validation_definitions.add_or_update(
        ValidationDefinition(
            name=f"val_{nombre_tabla}",
            data=batch_definition,
            suite=suite,
        )
    )

    checkpoint = context.checkpoints.add_or_update(
        Checkpoint(
            name=f"checkpoint_{nombre_tabla}",
            validation_definitions=[validation_definition],
        )
    )

    result = checkpoint.run(batch_parameters=batch_parameters)

    # --- Extraer resumen por expectativa, incluyendo la categoría asignada ---
    for validation_result in result.run_results.values():
        for r in validation_result.results:
            col_name = r.expectation_config.kwargs.get("column")
            resumen_resultados.append({
                "tabla": nombre_tabla,
                "columna": col_name,
                "categoria": clasificacion_columnas.get(col_name, "general"),
                "expectativa": r.expectation_config.type,
                "exito": r.success,
                "elementos_evaluados": r.result.get("element_count"),
                "no_cumplen": r.result.get("unexpected_count"),
                "pct_no_cumplen": r.result.get("unexpected_percent"),
            })

print("Listo. Resultados acumulados en 'resumen_resultados'.")

## 11. Resumen en un DataFrame (para revisar rápido en el notebook)

In [ ]:
df_resumen = pd.DataFrame(resumen_resultados)

# Solo lo que falló, ordenado por % de valores problemáticos
df_fallos = (
    df_resumen[df_resumen["exito"] == False]
    .sort_values("pct_no_cumplen", ascending=False)
)
df_fallos

## 12. Auditoría de la clasificación automática

Revisa qué categoría se asignó a cada columna, para detectar si alguna quedó mal
clasificada (ej. una columna de fecha que cayó en "personal") y ajustar
`COLUMN_CATEGORIES` si hace falta.

In [ ]:
df_resumen[["tabla", "columna", "categoria"]].drop_duplicates().sort_values(["tabla", "categoria"])

## 13. Generar el reporte HTML (Data Docs)

Abre el reporte navegable con el detalle de cada expectativa, columna y ejemplos de valores
que no cumplieron.

In [ ]:
context.build_data_docs()
context.open_data_docs()

## Notas

- **Carga sobre SQL Server**: se reduce a una sola query (`SELECT *`) por tabla — toda la
  validación posterior corre en memoria sobre el DataFrame, sin consultas adicionales al
  servidor.
- **Categorías**: ajusta `COLUMN_CATEGORIES` según la nomenclatura real de tus columnas.
  Puedes agregar categorías nuevas (ej. `"geografico"` para lat/lon) agregando una entrada
  al diccionario y una función en `EXPECTATION_BUILDERS`.
- **`mostly`**: cada categoría tolera un porcentaje distinto de excepciones — ajusta según
  qué tan estricto quieras ser por tipo de dato.
- **Memoria**: si alguna tabla es muy grande para caber cómoda como DataFrame de pandas,
  considera leerla en chunks (`pd.read_sql(..., chunksize=...)`) y acumular resultados, o
  usar Polars con modo lazy en su lugar.
- **Duplicados en "personal"**: intencionalmente no se exige `ExpectColumnValuesToBeUnique`
  en columnas personales — dos personas pueden llamarse igual. La unicidad se reserva para
  la categoría `"codigo"` (llaves, IDs).
